# 07 · Cómo se entrenó
*El schedule con warmup, label smoothing, dropout y Adam con β₂ = 0,98, implementados y medidos*

Notebook del episodio 7 de **Attention is all you need** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/attention-is-all-you-need/entrenamiento-y-legado)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** tabla de la receta → lr = d^-0,5 · min(s^-0,5, s·w^-1,5): cuál término gana a cada lado de w, pico 6,99e-4 en s = 4000, reescrita como pico · min(s/w, √(w/s)) → la pila post-norm de 12 capas del ep. 6 con warmup: 82 % → 97 % → label smoothing: objetivo 0,92/0,02, la pérdida mínima pasa a p = q (seguro 0,72 vs como q 0,39) → dropout: máscara Bernoulli y división por 1 − p, E[y] = x, nada en eval → Adam: v_t como promedio exponencial, memoria 1/(1 − β₂) = 50 pasos vs 1000; √v̂ se entera del cambio de escala → receta entera en nn.Transformer ordenando 10 dígitos: con ε_ls 0,1 baja la confianza (0,96 → 0,89) y sube la perplexity, igual acierto

**Cómo correrlo:** con el entorno de `~/Movies/serie-attention/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "attention"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Attention is all you need. Episodio siete: cómo se entrenó. Hoy, la receta para entrenarlo, la sección cinco entera: el learning rate que sube y baja, label smoothing, dropout y Adam, adaptive moment estimation, con un beta raro. Cada pieza en su fórmula, en código, y medida en una tarea de juguete.*

### La receta del paper (sección 5)

La receta, tal como está en el paper. El optimizador es Adam, con tres números que vamos a explicar. El learning rate no es fijo: sigue una fórmula, con un warmup de cuatro mil pasos. Dropout de cero coma uno. Label smoothing de cero coma uno. Cada batch junta frases hasta sumar unos veinticinco mil tokens de cada lado. Y el modelo base se entrenó cien mil pasos, doce horas, en ocho placas de video P100.

| Pieza | Valor en el paper |
|---|---|
| Optimizador | Adam · β₁ 0,9 · β₂ 0,98 · ε 10⁻⁹ |
| Learning rate | ecuación (3), warmup 4000 pasos |
| Dropout | P_drop = 0,1 (modelo base) |
| Label smoothing | εₗₛ = 0,1 |
| Batch | ≈ 25.000 tokens de entrada y 25.000 de salida |
| Entrenamiento | base: 100.000 pasos, 12 h, 8 placas P100 |

## Parte 1 · Un learning rate que sube y baja

La ecuación tres del paper. Primero la fórmula, despacio; después, para qué sirve la subida.

### La ecuación (3), término por término

$$
\mathit{lr}(s) = d_{\text{model}}^{-0.5} \cdot \min\big(s^{-0.5},\; s \cdot w^{-1.5}\big)
$$
<small>$s$: el paso · $w$: warmup_steps = 4000</small>

$$
s \cdot w^{-1.5} < s^{-0.5} \iff s^{1.5} < w^{1.5} \iff s < w
$$
<small>multiplicamos los dos lados por $s^{0.5}\, w^{1.5}$</small>

$$
\mathit{lr}(w) = (d_{\text{model}} \cdot w)^{-0.5} = 6{,}99 \cdot 10^{-4}
$$
<small>el pico, justo en $s = w$</small>

$$
\mathit{lr}(s) = \text{pico} \cdot \min\!\big(\tfrac{s}{w},\; \sqrt{w/s}\big)
$$
<small>la misma fórmula, reescrita</small>

s es el número de paso, w son los pasos de warmup, cuatro mil, y d sub model, quinientos doce. El mínimo entre dos términos: s a la menos un medio, que baja, y s por w a la menos uno coma cinco, una recta que sube. En el dibujo, en el paso mil la recta es más chica. Multiplicando los dos lados por s a la cero coma cinco y w a la uno coma cinco: la recta gana mientras s sea menor que w. En el paso dieciséis mil, manda el otro. El pico está justo en s igual a w: uno sobre la raíz de d por w, seis coma noventa y nueve por diez a la menos cuatro. Sacando el pico como factor, queda la forma: sube en línea recta hasta w y después baja como uno sobre raíz de s.

### La ecuación (3) en Python

La fórmula en una línea, con los valores del paper por defecto. La evaluamos en seis pasos. El pico, uno sobre la raíz de d por w. Y comprobamos la reescritura en los cien mil pasos. En el paso uno, uno coma setenta y cinco por diez a la menos siete: casi cero. En el paso mil, uno coma setenta y cinco por diez a la menos cuatro. En el cuatro mil, el pico. Y en sesenta y cuatro mil, otra vez uno coma setenta y cinco. La reescritura da True.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_lrate.py
__file__ = 'snippets/ep07_lrate.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math

In [3]:
def lrate(paso, d_model=512, warmup=4000):          # ecuación (3)
    return d_model ** -0.5 * min(paso ** -0.5, paso * warmup ** -1.5)

for paso in [1, 1000, 4000, 16000, 64000, 100000]:
    print(f"paso {paso:6}: lr = {lrate(paso):.2e}")

pico = (512 * 4000) ** -0.5                          # 1 / √(d · w)
forma = lambda s, w=4000: min(s / w, math.sqrt(w / s))
print(f"pico {pico:.2e} · igual a pico · forma:",
      all(abs(lrate(s) - pico * forma(s)) < 1e-12 for s in range(1, 10**5)))

paso      1: lr = 1.75e-07
paso   1000: lr = 1.75e-04
paso   4000: lr = 6.99e-04
paso  16000: lr = 3.49e-04
paso  64000: lr = 1.75e-04
paso 100000: lr = 1.40e-04
pico 6.99e-04 · igual a pico · forma: True


### El learning rate del modelo base, paso a paso

La curva entera. Los primeros cuatro mil pasos, una rampa: el learning rate arranca en casi cero y crece en línea recta. Después baja despacio, y al final, en el paso cien mil, sigue en uno coma cuatro por diez a la menos cuatro. Fijate que d sub model está en la fórmula: un modelo más ancho arranca con pasos más chicos.

> **Vaswani et al., 2017 — Attention Is All You Need, sección 5.3** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **warmup_steps = 4000** · «This corresponds to increasing the learning rate linearly for the first warmup_steps training steps, and decreasing it thereafter proportionally to the inverse square root of the step number.»

El paper lo dice en una frase. Subir linealmente durante los primeros warmup steps, y después bajar en proporción a la inversa de la raíz del paso. Con warmup steps igual a cuatro mil. No explica por qué. En el episodio seis vimos una razón: con post-norm, las capas de arriba reciben gradientes grandes al principio. La serie de Optimizadores, episodio ocho, muestra otra: Adam, al arrancar, divide por promedios de muy pocas muestras.

### La pila del episodio 6, con warmup

La prueba. La pila post-norm de doce capas del episodio seis, la que se trababa. La forma de la ecuación tres. El mismo pico, uno por diez a la menos tres, cuatrocientos pasos. LambdaLR multiplica el learning rate por lo que devuelve la función en cada paso. Sin warmup, y con cien pasos de warmup. El pico y w los elegimos a mano. Sin warmup, la pérdida se queda en uno coma veinticinco trescientos pasos, y al final llega a ochenta y dos por ciento. Con warmup, a los trescientos pasos ya bajó a cero coma treinta, y termina en noventa y siete.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_warmup.py
__file__ = 'snippets/ep07_warmup.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.set_num_threads(1)


def datos(n, T=10, V=10):
    """La tarea del episodio 6: para cada token, cuántas veces MÁS aparece (0..9)."""
    x = torch.randint(0, V, (n, T))
    return x, (x[:, :, None] == x[:, None, :]).sum(-1) - 1


def pila(N=12, d=32):
    """La pila post-norm de 12 capas del episodio 6 (misma inicialización)."""
    torch.manual_seed(0)
    capa = nn.TransformerEncoderLayer(d, 4, 4 * d, dropout=0.0, batch_first=True)
    return nn.Sequential(nn.Embedding(10, d), nn.TransformerEncoder(capa, N, enable_nested_tensor=False), nn.Linear(d, 10))

In [5]:
forma = lambda s, w: min(s / w, math.sqrt(w / s))   # la de la ecuación (3)

def entrenar(warmup, pico=1e-3, pasos=400):         # post-norm, 12 capas
    m = pila()
    opt = torch.optim.Adam(m.parameters(), lr=pico, betas=(0.9, 0.98), eps=1e-9)
    f = (lambda s: forma(s + 1, warmup)) if warmup else (lambda s: 1.0)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, f)
    torch.manual_seed(1); perdidas = []
    for paso in range(1, pasos + 1):
        x, y = datos(64)
        loss = F.cross_entropy(m(x).flatten(0, 1), y.flatten())
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
        if paso % 100 == 0: perdidas.append(f"{loss.item():.2f}")
    x, y = datos(2000)
    return perdidas, (m(x).argmax(-1) == y).float().mean().item()

for w in (0, 100):
    perdidas, acc = entrenar(w)
    print(f"warmup {w:3}: pérdida c/100 pasos", *perdidas, f"· {acc:.0%}")

warmup   0: pérdida c/100 pasos 1.25 1.25 1.25 0.38 · 82%


warmup 100: pérdida c/100 pasos 1.27 1.23 0.30 0.06 · 97%


### Pérdida, 12 capas post-norm (salida real)

En curvas. Sin warmup, una meseta en uno coma veinticinco: el modelo responde siempre cero, la respuesta más común. Se destraba recién después del paso trescientos. Con warmup, los primeros pasos son chiquitos, y cuando el learning rate llega al pico, los pesos ya no son ruido. Misma red, mismo pico, cien pasos de rampa.

## Parte 2 · Label smoothing

La segunda pieza cambia lo que el modelo tiene que acertar: en vez de un uno y todos ceros, algo un poco más suave.

### El objetivo suavizado, con 5 clases

$$
y = (1, 0, 0, 0, 0)
$$
<small>one-hot: la clase correcta es la 0</small>

$$
q_k = (1 - \varepsilon)\, y_k + \frac{\varepsilon}{V}
$$
<small>$\varepsilon = 0{,}1$, $V = 5$: $0{,}92$ y $0{,}02$</small>

$$
\mathcal{L} = -\textstyle\sum_k q_k \log p_k
$$
<small>la cross-entropy contra $q$</small>

$$
\min_p \mathcal{L} \;\text{ en }\; p = q
$$
<small>lo mejor ya no es estar 100 % seguro</small>

Cinco clases, y la correcta es la cero. El objetivo de siempre, y, es one-hot: un uno en la correcta y ceros en el resto. Label smoothing lo mezcla con la distribución uniforme: q sub k es uno menos épsilon por y sub k, más épsilon sobre V, donde V es la cantidad de clases. Con épsilon cero coma uno: cero coma noventa y dos para la correcta y cero coma cero dos para cada otra. La pérdida es la cross-entropy contra q: menos la suma de q sub k por el logaritmo de p sub k, la probabilidad que da el modelo. En el dibujo, un modelo seguro: las otras clases tienen probabilidad casi cero, su logaritmo es muy negativo, y cada una cobra cero coma dieciocho. La pérdida mínima es cuando p es igual a q: el modelo ya no gana nada con estar cien por ciento seguro.

### Label smoothing en PyTorch

El objetivo suavizado: épsilon sobre V en todas las clases, y la correcta suma uno menos épsilon. Tres modelos: uno seguro, uno que predice exactamente q, y uno uniforme. PyTorch lo trae: el argumento label smoothing de cross entropy. Y la cuenta a mano, para comparar. El objetivo, cero coma noventa y dos y cuatro de cero coma cero dos. Sin suavizar, el seguro gana con pérdida cero. Con épsilon, el seguro paga cero coma setenta y dos y el que predice q, cero coma treinta y nueve: ahora gana el menos seguro. Y la cuenta a mano coincide con PyTorch en los tres.

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_suavizado.py
__file__ = 'snippets/ep07_suavizado.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn.functional as F

In [7]:
V, eps, y = 5, 0.1, torch.tensor([0])            # 5 clases, la correcta es la 0
q = torch.full((V,), eps / V); q[0] += 1 - eps   # objetivo suavizado
print("objetivo:", [round(v, 2) for v in q.tolist()])

modelos = {"seguro":   torch.tensor([[9., 0, 0, 0, 0]]),
           "como q":   q.log().unsqueeze(0),      # predice exactamente q
           "uniforme": torch.zeros(1, V)}
for nombre, logits in modelos.items():
    p = logits.softmax(-1)[0]
    duro = F.cross_entropy(logits, y)
    suave = F.cross_entropy(logits, y, label_smoothing=eps)
    a_mano = -(q * p.log()).sum()
    print(f"{nombre:8} p0={p[0]:.2f}  sin ε {duro:.2f}  con ε {suave:.2f}"
          f"  a mano {a_mano:.2f}")

objetivo: [0.92, 0.02, 0.02, 0.02, 0.02]
seguro   p0=1.00  sin ε 0.00  con ε 0.72  a mano 0.72
como q   p0=0.92  sin ε 0.08  con ε 0.39  a mano 0.39
uniforme p0=0.20  sin ε 1.61  con ε 1.61  a mano 1.61


> **Vaswani et al., 2017 — Attention Is All You Need, sección 5.4** · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
>
> **εₗₛ = 0,1** · «This hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score.»

El paper lo reconoce. Label smoothing empeora la perplexity, porque el modelo aprende a estar menos seguro, pero mejora el acierto y el BLEU, bilingual evaluation understudy, la métrica de traducción que vemos en el episodio ocho. La perplexity es e a la pérdida: si vale uno, el modelo está seguro; si vale cinco, duda como si eligiera entre cinco opciones. En el paper, V son unos treinta y siete mil tokens: cada token equivocado recibe cero coma uno sobre treinta y siete mil.

## Parte 3 · Dropout y Adam

Dos piezas más: una apaga números al azar mientras entrena, la otra decide el tamaño de cada paso.

### Dropout, con ocho números

$$
m_i \sim \mathrm{Bernoulli}(1 - p)
$$
<small>cada número sobrevive con probabilidad $1 - p = 0{,}9$</small>

$$
y_i = \frac{m_i\, x_i}{1 - p}
$$
<small>los que quedan se agrandan: $1/0{,}9 = 1{,}11$</small>

$$
\mathbb{E}[y_i] = (1 - p) \cdot \frac{x_i}{1 - p} = x_i
$$
<small>en promedio no cambia nada; en eval, $y = x$</small>

Dropout, que en el episodio seis apareció en cada subcapa. p es la probabilidad de apagar: cero coma uno. Para cada número se tira una moneda, m sub i: vale uno con probabilidad cero coma nueve y cero si no. En el dibujo, la máscara que salió: se apagaron el primero y el cuarto. Los que sobreviven se dividen por uno menos p: si x vale uno, quedan en uno coma once. ¿Por qué dividir? La esperanza, el promedio sobre muchas tiradas, es cero coma nueve por x sobre cero coma nueve: x. Así la escala es la misma al entrenar que al evaluar, cuando dropout no hace nada. Con ocho números, el promedio da cero coma ochenta y tres; con cien mil, uno.

### nn.Dropout en train y en eval

Dropout con la p del paper. En modo train. Con cien mil números, para ver la proporción. Y en modo eval. En train, dos ceros y el resto en uno coma once. Con cien mil, se apagó el diez coma uno por ciento y el promedio es cero coma novecientos noventa y nueve. En eval, todo queda en uno. Por eso, model punto eval antes de medir.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_dropout.py
__file__ = 'snippets/ep07_dropout.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn as nn
torch.manual_seed(0)

In [9]:
drop = nn.Dropout(p=0.1)                     # P_drop del paper
x = torch.ones(8)

drop.train()
print("train:", [round(v, 2) for v in drop(x).tolist()])
grande = drop(torch.ones(100_000))
print(f"apagados {(grande == 0).float().mean():.1%} · "
      f"promedio {grande.mean():.3f}")

drop.eval()
print("eval: ", drop(x).tolist())

train: [0.0, 1.11, 1.11, 0.0, 1.11, 1.11, 1.11, 1.11]
apagados 10.1% · promedio 0.999
eval:  [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


### Adam y su β₂ = 0,98

$$
v_t = \beta_2\, v_{t-1} + (1 - \beta_2)\, g_t^2
$$
<small>un promedio del gradiente al cuadrado</small>

$$
\text{peso de } g_{t-k}^2 = (1 - \beta_2)\, \beta_2^{\,k}
$$
<small>cuánto pesa un gradiente de hace $k$ pasos</small>

$$
\textstyle\sum_{k \ge 0} \beta_2^{\,k} = \frac{1}{1 - \beta_2}
$$
<small>la memoria: 50 pasos con 0,98; 1000 con 0,999</small>

$$
\Delta w = -\eta\, \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}
$$
<small>el paso se divide por la escala que recuerda $v$</small>

Adam lo armamos en Optimizadores, episodio cinco; acá, solo lo que cambia el paper. v sub t es un promedio del gradiente al cuadrado, g sub t al cuadrado. Cada paso guarda cero coma noventa y ocho de lo que tenía y suma cero coma cero dos del gradiente nuevo. Un gradiente de hace k pasos pesa uno menos beta dos, por beta dos a la k: en el dibujo, a los doscientos pasos ya casi no cuenta. Sumando todos los pesos, la memoria es uno sobre uno menos beta dos: cincuenta pasos, contra mil con el valor por defecto, cero coma novecientos noventa y nueve. Adam divide cada paso por la raíz de v sombrero, v corregido para que no arranque en cero, más un épsilon. Si v tiene memoria corta, se entera rápido cuando el gradiente cambia de escala.

### Qué cambia β₂, medido

Mil pasos con gradiente uno, y de golpe cae a cero coma uno. El promedio de Adam. La corrección de sesgo. Los dos betas. Con cero coma noventa y ocho, la escala estimada baja a cero coma sesenta y uno a los cincuenta pasos, y a cero coma diecisiete a los doscientos: se está enterando. Con cero coma novecientos noventa y nueve, a los doscientos pasos todavía cree que el gradiente vale cero coma ochenta y seis, y Adam da pasos ocho veces más chicos de lo que debería. El paper no explica por qué eligió cero coma noventa y ocho; esto es lo que esa elección cambia.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_adam.py
__file__ = 'snippets/ep07_adam.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math

In [11]:
def escala(beta2, chicos=(10, 50, 200)):
    v, out = 0.0, []
    g = [1.0] * 1000 + [0.1] * max(chicos)    # el gradiente cae 10×
    for t, gi in enumerate(g, 1):
        v = beta2 * v + (1 - beta2) * gi ** 2  # promedio de g², como Adam
        v_hat = v / (1 - beta2 ** t)           # corrección de sesgo
        if t - 1000 in chicos:
            out.append(f"{math.sqrt(v_hat):.2f}")
    return out

for b2 in (0.98, 0.999):
    print(f"β2 = {b2:<5} (memoria ≈ {1 / (1 - b2):4.0f} pasos):",
          "√v̂ tras 10, 50, 200:", *escala(b2))
print("escala real del gradiente: 0.10")

β2 = 0.98  (memoria ≈   50 pasos): √v̂ tras 10, 50, 200: 0.90 0.61 0.17
β2 = 0.999 (memoria ≈ 1000 pasos): √v̂ tras 10, 50, 200: 0.99 0.96 0.86
escala real del gradiente: 0.10


## Parte 4 · La receta entera

Todo junto, en el transformer chico de nn.Transformer, aprendiendo a ordenar diez dígitos.

### Entrenar con la receta del paper

La tarea: entra una secuencia de diez dígitos y el decoder tiene que escribirla ordenada. El modelo: nn.Transformer, como en el episodio cinco, más chico. Adam con los betas y el épsilon del paper. El schedule con la forma de la ecuación tres. Label smoothing. Y dropout de cero coma uno. Dos corridas, con y sin label smoothing, midiendo con la respuesta correcta como entrada del decoder. Con épsilon cero coma uno: noventa y siete coma siete por ciento de secuencias enteras bien, confianza media cero coma ochenta y nueve, perplexity uno coma ciento veinticinco. Sin suavizar: noventa y siete coma cuatro, confianza cero coma noventa y seis, perplexity uno coma cero cuarenta y siete.

In [12]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep07_receta.py
__file__ = 'snippets/ep07_receta.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import torch
import torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from _ordenar import Ordenador, datos, evaluar, forma   # tarea: ordenar 10 dígitos

In [13]:
def receta(modelo, eps_ls, pasos=800, pico=2e-3, warmup=100):
    opt = torch.optim.Adam(modelo.parameters(), lr=pico,
                           betas=(0.9, 0.98), eps=1e-9)   # sección 5.3
    sch = LambdaLR(opt, lambda s: forma(s + 1, warmup))  # ecuación (3)
    modelo.train()                                       # dropout prendido
    for _ in range(pasos):
        x, y, y_in = datos(64)
        loss = F.cross_entropy(modelo(x, y_in).flatten(0, 1), y.flatten(),
                               label_smoothing=eps_ls)    # sección 5.4
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    return modelo

for eps_ls in (0.1, 0.0):
    torch.manual_seed(0)
    r = evaluar(receta(Ordenador(p=0.1), eps_ls))      # P_drop = 0,1
    print(f"ε_ls {eps_ls}: secuencias {r['secuencia']:.1%} · confianza "
          f"{r['confianza']:.2f} · perplejidad {r['perplejidad']:.3f}")

ε_ls 0.1: secuencias 97.7% · confianza 0.89 · perplejidad 1.125


ε_ls 0.0: secuencias 97.4% · confianza 0.96 · perplejidad 1.047


### Con y sin label smoothing (salida real)

Lo que dijo el paper, en chico. El acierto, prácticamente igual: en una tarea tan fácil no hay mucho para mejorar. La confianza baja: el modelo aprendió a dejar algo de probabilidad para las otras clases. Y la perplexity sube: empeora, como avisa el paper. En traducción, con muchas frases correctas posibles, ese modelo menos seguro rinde más BLEU.

|  | εₗₛ = 0,1 | εₗₛ = 0 |
|---|---|---|
| Secuencias enteras bien | 97,7 % | 97,4 % |
| Confianza media | 0,89 | 0,96 |
| Perplexity | 1,125 | 1,047 |

## Resumiendo

- $\mathit{lr} = d^{-0.5} \min(s^{-0.5}, s\, w^{-1.5})$: pico $6{,}99 \cdot 10^{-4}$ en $s = 4000$
- Warmup: 12 capas post-norm, 82 % → 97 %
- Label smoothing: objetivo 0,92 / 0,02; gana el modelo menos seguro
- Dropout: apaga el 10 % y divide por 0,9; en eval no hace nada
- Adam con β₂ = 0,98: memoria de 50 pasos en vez de 1000
- Próximo: resultados, BLEU y ablaciones